# SpaceX Falcon 9 First Stage Landing Prediction
## IBM Applied Data Science Capstone Project

**Project goal:** Explore historical Falcon 9 launch data and build a machine-learning model to estimate whether the first stage will land successfully.

> **Important:** Run the notebook from top to bottom in Google Colab or Jupyter Notebook. The data-collection cells need an internet connection. The charts, SQL results, and model metrics are generated from the data retrieved when you run the notebook; do not report results until those cells finish successfully.

### Project workflow
1. Collect launch records from the SpaceX API.
2. Scrape a launch table from Wikipedia as a secondary source.
3. Clean and wrangle the data.
4. Explore the data using visualizations and SQL.
5. Display launch sites on a Folium map.
6. Train and evaluate classification models.


## 1. Install and import libraries

If a package is already installed, pip will leave it in place. In Google Colab, run this cell first.

In [ ]:
%pip -q install pandas numpy requests beautifulsoup4 lxml matplotlib seaborn plotly folium scikit-learn sqlalchemy

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import folium

from bs4 import BeautifulSoup
from IPython.display import display
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

## 2. Collect launch data from the SpaceX API

The SpaceX API provides launch details including launch dates, rocket names, payloads, launch sites, and landing outcomes. This cell requests the launch collection and normalizes the JSON response into a table.

In [ ]:
SPACEX_LAUNCHES_URL = "https://api.spacexdata.com/v4/launches"

response = requests.get(SPACEX_LAUNCHES_URL, timeout=30)
response.raise_for_status()
launches_json = response.json()

print(f"Launch records downloaded: {len(launches_json)}")
print("Example fields:", list(launches_json[0].keys())[:15] if launches_json else "No records returned")

In [ ]:
# Normalize the API response. Keep nested fields for extracting in later steps.
raw_launches = pd.json_normalize(launches_json, sep="_")
display(raw_launches.head())
print("Raw shape:", raw_launches.shape)

## 3. Build a modeling dataset

The target is `landing_success`:
- `1`: the core landing outcome is recorded as successful.
- `0`: the core landing outcome is not recorded as successful.

This is a simplified historical classification target. Real launch operations can have missing, reused, or non-applicable landing outcomes, so this notebook filters to records with an identifiable first-stage/core landing outcome. Payloads and mission details are extracted from the API's nested records.

In [ ]:
# Fetch core and payload details for joins with the launch collection.
CORES_URL = "https://api.spacexdata.com/v4/cores"
PAYLOADS_URL = "https://api.spacexdata.com/v4/payloads"
ROCKETS_URL = "https://api.spacexdata.com/v4/rockets"
LAUNCHPADS_URL = "https://api.spacexdata.com/v4/launchpads"

def fetch_json(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

cores_json = fetch_json(CORES_URL)
payloads_json = fetch_json(PAYLOADS_URL)
rockets_json = fetch_json(ROCKETS_URL)
launchpads_json = fetch_json(LAUNCHPADS_URL)

cores_by_id = {item.get("id"): item for item in cores_json}
payloads_by_id = {item.get("id"): item for item in payloads_json}
rockets_by_id = {item.get("id"): item for item in rockets_json}
launchpads_by_id = {item.get("id"): item for item in launchpads_json}

records = []
for launch in launches_json:
    core_entries = launch.get("cores") or []
    if not core_entries:
        continue

    core_entry = core_entries[0] or {}
    core_id = core_entry.get("core")
    core = cores_by_id.get(core_id, {}) if core_id else {}

    # Prefer the outcome attached to the launch's core record, then the core's latest landing outcome.
    landing_outcome = core_entry.get("landing_success")
    if landing_outcome is None:
        landing_outcome = core.get("last_update")
        # last_update is descriptive text, not a binary label; leave unknown outcomes out.
        landing_outcome = None

    # For older/newer API records, attempt to derive success from core landing history.
    if landing_outcome is None:
        landings = core.get("landings")
        if isinstance(landings, list) and len(landings) > 0:
            # A list of landing attempts alone does not establish that this launch landed successfully.
            landing_outcome = None

    payload_ids = launch.get("payloads") or []
    payload_records = [payloads_by_id.get(pid, {}) for pid in payload_ids]
    masses = [p.get("mass_kg") for p in payload_records if isinstance(p.get("mass_kg"), (int, float))]
    payload_mass_kg = float(sum(masses)) if masses else np.nan

    rocket = rockets_by_id.get(launch.get("rocket"), {})
    launchpad = launchpads_by_id.get(launch.get("launchpad"), {})

    records.append({
        "launch_id": launch.get("id"),
        "flight_number": launch.get("flight_number"),
        "name": launch.get("name"),
        "date_utc": launch.get("date_utc"),
        "date_year": pd.to_datetime(launch.get("date_utc"), errors="coerce", utc=True).year,
        "rocket_id": launch.get("rocket"),
        "rocket_name": rocket.get("name"),
        "launchpad_id": launch.get("launchpad"),
        "launchpad_name": launchpad.get("name"),
        "launchpad_full_name": launchpad.get("full_name"),
        "latitude": launchpad.get("latitude"),
        "longitude": launchpad.get("longitude"),
        "payload_mass_kg": payload_mass_kg,
        "payload_count": len(payload_ids),
        "core_id": core_id,
        "core_reused": core_entry.get("reused"),
        "landing_attempt": core_entry.get("landing_attempt"),
        "landing_type": core_entry.get("landing_type"),
        "landing_success": landing_outcome,
        "launch_success": launch.get("success"),
        "details": launch.get("details")
    })

launch_df = pd.DataFrame(records)
print("Constructed launch table shape:", launch_df.shape)
display(launch_df.head())

### Important data note

The SpaceX API schema can change and some launch records do not expose a reliable binary landing outcome for that specific core attempt. The next cell checks whether the retrieved data has usable labels. If the API's current records do not include enough labels, the notebook will stop the modeling section with an explanatory message rather than invent results.

In [ ]:
# Keep records with a definite boolean landing label.
model_df = launch_df[launch_df["landing_success"].apply(lambda x: isinstance(x, (bool, np.bool_)))].copy()
model_df["landing_success"] = model_df["landing_success"].astype(int)

print("Rows with explicit landing labels:", len(model_df))
if model_df.empty:
    print(
        "No launch-specific binary landing labels were available in the current API response. "
        "The API schema may have changed or this endpoint may not provide historical attempt outcomes. "
        "Do not invent model metrics. Use the IBM course dataset / supplied capstone CSVs to complete "
        "the supervised-learning section, or adapt the extraction to a verified landing-outcome source."
    )
else:
    display(model_df[["flight_number", "name", "date_utc", "launchpad_name",
                      "payload_mass_kg", "landing_type", "landing_success"]].head())
    print(model_df["landing_success"].value_counts(dropna=False))

## 4. Web scraping: Wikipedia launch table

This section retrieves a Falcon 9 launch table from Wikipedia. The page layout can change, so the code lists the tables found and selects the first table that appears to contain launch-related columns. Use this as a secondary source and cite the page in your report.

In [ ]:
WIKIPEDIA_URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
wiki_response = requests.get(
    WIKIPEDIA_URL,
    headers={"User-Agent": "Mozilla/5.0 (compatible; educational-data-project/1.0)"},
    timeout=30
)
wiki_response.raise_for_status()
wiki_tables = pd.read_html(wiki_response.text)

print("Tables detected:", len(wiki_tables))
for i, table in enumerate(wiki_tables[:10]):
    print(i, table.shape, list(table.columns)[:6])

In [ ]:
# Select a likely launch-history table based on its column names.
wiki_launches = None
for table in wiki_tables:
    cols = " ".join(str(c).lower() for c in table.columns)
    if "launch" in cols and ("date" in cols or "payload" in cols or "booster" in cols):
        wiki_launches = table.copy()
        break

if wiki_launches is None and wiki_tables:
    wiki_launches = max(wiki_tables, key=len).copy()

if wiki_launches is not None:
    print("Selected scraped table shape:", wiki_launches.shape)
    display(wiki_launches.head())
else:
    print("No tables were found. Check the page structure or use the course-provided scraped dataset.")

## 5. Data wrangling and quality checks

Inspect missing values, data types, duplicates, and key distributions before analysis.

In [ ]:
print("Launch dataset shape:", launch_df.shape)
display(launch_df.dtypes.to_frame("dtype"))
display(launch_df.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(15))
print("Duplicate launch IDs:", launch_df["launch_id"].duplicated().sum())

In [ ]:
# Standardize and prepare analysis columns.
launch_df["date_utc"] = pd.to_datetime(launch_df["date_utc"], errors="coerce", utc=True)
launch_df["date_year"] = launch_df["date_utc"].dt.year
launch_df["payload_mass_kg"] = pd.to_numeric(launch_df["payload_mass_kg"], errors="coerce")
launch_df["payload_count"] = pd.to_numeric(launch_df["payload_count"], errors="coerce")

analysis_df = launch_df.drop_duplicates(subset=["launch_id"]).copy()
display(analysis_df[["flight_number", "name", "date_utc", "date_year",
                     "launchpad_name", "payload_mass_kg", "launch_success"]].head(10))

## 6. Exploratory data analysis (EDA)

### Launches by year

In [ ]:
year_counts = analysis_df.dropna(subset=["date_year"]).groupby("date_year").size().reset_index(name="launch_count")
if not year_counts.empty:
    fig = px.bar(year_counts, x="date_year", y="launch_count",
                 title="SpaceX Launches by Year", labels={"date_year": "Year", "launch_count": "Number of launches"})
    fig.show()
else:
    print("No valid launch years available.")

### Payload mass distribution

In [ ]:
payload_data = analysis_df.dropna(subset=["payload_mass_kg"])
if not payload_data.empty:
    fig = px.histogram(payload_data, x="payload_mass_kg", nbins=25,
                       title="Distribution of Payload Mass", labels={"payload_mass_kg": "Payload mass (kg)"})
    fig.show()
else:
    print("Payload mass data is not available in the current API response.")

### Launch success over time

In [ ]:
success_df = analysis_df.dropna(subset=["launch_success", "date_year"]).copy()
if not success_df.empty:
    yearly_success = success_df.groupby("date_year")["launch_success"].mean().reset_index()
    yearly_success["success_rate_pct"] = yearly_success["launch_success"] * 100
    fig = px.line(yearly_success, x="date_year", y="success_rate_pct", markers=True,
                  title="Launch Success Rate by Year",
                  labels={"date_year": "Year", "success_rate_pct": "Success rate (%)"})
    fig.show()
    display(yearly_success[["date_year", "success_rate_pct"]])
else:
    print("Launch-success values are missing for the available records.")

### Payload mass versus launch outcome

In [ ]:
outcome_df = analysis_df.dropna(subset=["payload_mass_kg", "launch_success"]).copy()
if not outcome_df.empty:
    outcome_df["launch_outcome"] = outcome_df["launch_success"].map({True: "Success", False: "Failure"})
    fig = px.scatter(outcome_df, x="payload_mass_kg", y="flight_number", color="launch_outcome",
                     hover_name="name", title="Payload Mass and Launch Outcome",
                     labels={"payload_mass_kg": "Payload mass (kg)", "flight_number": "Flight number"})
    fig.show()
else:
    print("Not enough payload and outcome data for this scatter plot.")

## 7. SQL analysis with SQLite

Load the collected launch data into an in-memory SQLite database and use SQL queries to explore launch sites, payloads, outcomes, and yearly activity.

In [ ]:
sql_df = analysis_df.copy()
sql_df["date_year"] = pd.to_numeric(sql_df["date_year"], errors="coerce")
sql_df["payload_mass_kg"] = pd.to_numeric(sql_df["payload_mass_kg"], errors="coerce")
engine = create_engine("sqlite://")
sql_df.to_sql("launches", engine, index=False, if_exists="replace")
print("SQLite table 'launches' created.")

In [ ]:
query = '''
SELECT launchpad_name,
       COUNT(*) AS total_launches,
       SUM(CASE WHEN launch_success = 1 THEN 1 ELSE 0 END) AS successful_launches,
       ROUND(100.0 * AVG(CASE WHEN launch_success = 1 THEN 1.0 ELSE 0.0 END), 2) AS success_rate_pct
FROM launches
GROUP BY launchpad_name
ORDER BY total_launches DESC
'''
display(pd.read_sql_query(query, engine))

In [ ]:
query = '''
SELECT date_year, COUNT(*) AS launches,
       SUM(CASE WHEN launch_success = 1 THEN 1 ELSE 0 END) AS successful_launches
FROM launches
WHERE date_year IS NOT NULL
GROUP BY date_year
ORDER BY date_year
'''
display(pd.read_sql_query(query, engine))

In [ ]:
query = '''
SELECT launchpad_name,
       ROUND(AVG(payload_mass_kg), 2) AS average_payload_kg,
       MAX(payload_mass_kg) AS maximum_payload_kg,
       COUNT(payload_mass_kg) AS launches_with_payload_data
FROM launches
GROUP BY launchpad_name
ORDER BY average_payload_kg DESC
'''
display(pd.read_sql_query(query, engine))

## 8. Interactive launch-site map with Folium

Plot launch pads using coordinates returned by the SpaceX API. Each marker includes the launch site name and the number of launch records associated with that site.

In [ ]:
site_map_df = (analysis_df.dropna(subset=["latitude", "longitude"])
               .groupby(["launchpad_id", "launchpad_name", "launchpad_full_name", "latitude", "longitude"], dropna=False)
               .size().reset_index(name="launch_count"))

if not site_map_df.empty:
    center = [site_map_df["latitude"].mean(), site_map_df["longitude"].mean()]
    launch_map = folium.Map(location=center, zoom_start=3, tiles="OpenStreetMap")
    for _, row in site_map_df.iterrows():
        popup = (
            f"<b>{row['launchpad_full_name'] or row['launchpad_name']}</b><br>"
            f"Launch records: {int(row['launch_count'])}<br>"
            f"Coordinates: {row['latitude']:.4f}, {row['longitude']:.4f}"
        )
        folium.Marker(
            location=[row["latitude"], row["longitude"]],
            tooltip=str(row["launchpad_name"]),
            popup=folium.Popup(popup, max_width=300),
            icon=folium.Icon(color="blue", icon="info-sign")
        ).add_to(launch_map)
    display(launch_map)
else:
    print("Launch-site coordinates were not available.")

## 9. Interactive Plotly charts

These charts support interactive exploration. Inspect hover labels and use them to describe patterns only after running the notebook.

In [ ]:
if not year_counts.empty:
    fig = px.scatter(year_counts, x="date_year", y="launch_count", size="launch_count",
                     title="Yearly Launch Activity (Interactive)",
                     labels={"date_year": "Year", "launch_count": "Launch count"})
    fig.show()

site_counts = analysis_df["launchpad_name"].fillna("Unknown").value_counts().reset_index()
site_counts.columns = ["launch_site", "launch_count"]
if not site_counts.empty:
    fig = px.bar(site_counts, x="launch_site", y="launch_count",
                 title="Launch Records by Launch Site",
                 labels={"launch_site": "Launch site", "launch_count": "Launch records"})
    fig.show()

## 10. Machine-learning model: first-stage landing prediction

This section trains several classifiers only when the retrieved data includes enough explicit launch-specific landing labels and both classes are represented. The current SpaceX API may not expose a directly usable historical landing label in the expected field. If labels are unavailable, use the labelled dataset provided by your course and update the feature-column names below; do not fabricate labels.

The model uses features available before a launch, where present: payload mass, payload count, flight number, year, booster reuse, landing attempt, and launch site.

In [ ]:
if model_df.empty:
    print("Model training skipped: no verified binary landing labels were retrieved. Use the labelled course dataset to run this section.")
else:
    ml_df = model_df.copy()
    ml_df["date_utc"] = pd.to_datetime(ml_df["date_utc"], errors="coerce", utc=True)
    ml_df["date_year"] = ml_df["date_utc"].dt.year
    ml_df["core_reused"] = ml_df["core_reused"].map({True: 1, False: 0})
    ml_df["landing_attempt"] = ml_df["landing_attempt"].map({True: 1, False: 0})
    feature_cols_num = ["payload_mass_kg", "payload_count", "flight_number", "date_year",
                        "core_reused", "landing_attempt"]
    for col in feature_cols_num:
        if col not in ml_df.columns:
            ml_df[col] = np.nan
    feature_df = ml_df[feature_cols_num + ["launchpad_name", "landing_success"]].copy()
    feature_df = feature_df.dropna(subset=["landing_success"])
    feature_df[feature_cols_num] = feature_df[feature_cols_num].fillna(feature_df[feature_cols_num].median(numeric_only=True))
    feature_df["launchpad_name"] = feature_df["launchpad_name"].fillna("Unknown")
    feature_df = pd.get_dummies(feature_df, columns=["launchpad_name"], drop_first=True)
    X = feature_df.drop(columns=["landing_success"])
    y = feature_df["landing_success"].astype(int)

    if y.nunique() < 2 or y.value_counts().min() < 2:
        print("Model training skipped: both outcome classes need at least two examples.")
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )
        models = {
            "Logistic Regression": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=2000))]),
            "Support Vector Machine": Pipeline([("scale", StandardScaler()), ("model", SVC())]),
            "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
            "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
        }
        results = []
        fitted_models = {}
        for name, estimator in models.items():
            estimator.fit(X_train, y_train)
            pred = estimator.predict(X_test)
            fitted_models[name] = estimator
            results.append({
                "model": name,
                "accuracy": accuracy_score(y_test, pred),
                "precision": precision_score(y_test, pred, zero_division=0),
                "recall": recall_score(y_test, pred, zero_division=0),
                "f1_score": f1_score(y_test, pred, zero_division=0)
            })
        metrics_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
        display(metrics_df)
        best_name = metrics_df.iloc[0]["model"]
        best_model = fitted_models[best_name]
        best_predictions = best_model.predict(X_test)
        print("Best model by test accuracy:", best_name)
        print(classification_report(y_test, best_predictions, zero_division=0))
        cm = confusion_matrix(y_test, best_predictions)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title(f"Confusion Matrix — {best_name}")
        plt.xlabel("Predicted label")
        plt.ylabel("True label")
        plt.show()

## 11. Results and conclusion

After running the notebook, write your conclusion using the actual tables and model metrics produced above. Answer these questions:

1. Which launch site has the most launch records in the retrieved data?
2. How has annual launch activity changed over time?
3. What payload-mass patterns can you observe?
4. Which machine-learning model performs best on the held-out test data, and what are its accuracy, precision, recall, and F1-score?
5. What are the limitations of this analysis (missing labels, incomplete fields, API schema changes, and historical data coverage)?

**Do not claim a model result if model training was skipped.** If the API does not provide enough verified landing labels, use the labelled dataset from your IBM course materials and rerun the modeling section.

### Data sources
- SpaceX API: https://github.com/r-spacex/SpaceX-API
- SpaceX API launches endpoint: https://api.spacexdata.com/v4/launches
- Wikipedia Falcon 9 and Falcon Heavy launch history: https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches

### Submission checklist
- [ ] Run all cells from top to bottom.
- [ ] Check that the API and scraping cells completed successfully.
- [ ] Save the notebook with outputs after running it.
- [ ] Replace the report's placeholder figures and metrics with actual results.
- [ ] Add the notebook and final PDF to your GitHub repository.
- [ ] Copy the repository URL into the assignment submission.
